## 요약

물류 운영팀이 세 가지 배송 경로배정 전략(정적 기준선, 동적 재경로, AI 최적화)을 세 개의 배송거점 지역에 걸쳐 비교하는 무작위 현장시험을 계획하며, 평균 배송 지연(분)을 반응변수로 삼습니다. PROC GLMPOWER는 예상 셀 평균값으로 구성된 *예시* 데이터셋을 이용해, 전략 효과를 80%와 90% 검정력으로 검출하는 데 필요한 총 기사 수를 계산한 뒤, 경로 간 변동성이 커짐에 따라 달성 가능한 검정력이 어떻게 낮아지는지를 보여줍니다.

# PROC GLMPOWER를 이용한 기사 경로배정 현장시험 규모 산정

## 요약

물류 운영팀이 세 가지 배송 경로배정 전략 -- **정적(Static)** 기준선, **동적(Dynamic)** 재경로 시스템, **AI 최적화(AI-Optimized)** 플래너 -- 를 세 개의 배송거점 지역(북동부, 중서부, 서부)에 걸쳐 비교하는 무작위 현장시험을 막 시작하려 합니다. 반응변수는 평균 **배송 지연(분)** 입니다. 시험에 차량 자원을 투입하기 전에, 팀은 다음 질문에 답해야 합니다: *예상하는 운영 개선 효과를 안정적으로 검출하려면 기사가 몇 명 필요한가?*

이 노트북은 **PROC GLMPOWER**를 사용해 시험 배경의 일반선형모형에 대한 사전 검정력 및 표본크기 분석을 수행합니다. 예상 셀 평균값으로 구성된 *예시* 데이터셋을 바탕으로, (1) 전략 및 지역 전체효과에 대해 80%와 90% 검정력에 도달하는 데 필요한 총 등록 인원을 산출하고, (2) 경로 간 변동성이 커짐에 따라 달성 가능한 검정력이 어떻게 저하되는지 매핑하며, (3) 등록 인원 결정을 뒷받침할 검정력 곡선을 산출합니다.

> **핵심 결론:** 배송 지연의 오차 표준편차가 8분이라는 그럴듯한 가정 하에, 약 **63명의 기사**가 경로배정 전략 효과 검출에 80% 검정력을, **83명의 기사**가 90% 검정력을 제공합니다 -- 하지만 지연 변동성이 10분에 가까워지면 90명의 기사를 등록해도 70% 검정력에 미달합니다. 이는 일관되고 계측이 잘 된 경로에 투자하는 것의 가치를 강조합니다.

---
## 1. 예시 설계 구성

첫 단계는 시험 구조를 인코딩하고, 각 경로배정 전략 x 배송거점 지역 조합에 대해 팀이 *예상하는* 평균 지연값을 부여합니다. 난수 시드를 고정하고 미세한 흔들림(jitter)을 더해, 평균값이 현실적으로 보이면서도 3x3 균형 구조는 유지되도록 합니다. 모든 셀의 `cell_n` 가중치 1은 GLMPOWER에게 이 설계가 균형 설계임을 알려줍니다.

In [1]:
/* ================================================================
   예상 셀 평균 지연으로 이루어진 예시 데이터셋 생성.
   경로배정 전략 x 배송거점 지역 설계 셀당 한 행.
   ================================================================ */
데이터 routing_trial;
   호출 streaminit(20260531);
   길이 routing_strategy $8 depot_region $9;
   배열 strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   배열 region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   배열 smean[3]     _temporary_ (18.0 14.5 11.0);   /* 예상 전략 평균 */
   배열 radj[3]      _temporary_ (1.5  0.0 -1.0);    /* 지역 조정값(분)  */
   반복 i = 1 까지 3;
      반복 j = 1 까지 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         출력;
      종료;
   종료;
   제거 i j;
실행;

처리 인쇄 데이터=routing_trial 라벨 noobs;
   변수 routing_strategy depot_region mean_delay_min cell_n;
   라벨 routing_strategy="경로배정 전략" depot_region="배송거점 지역"
         mean_delay_min="평균 배송 지연(분)" cell_n="셀 가중치";
   제목 "예시 셀 평균값: 예상 배송 지연(분)";
실행;

                                                 예시 셀 평균값: 예상 배송 지연(분)                                                  

            경로배정 전략              배송거점 지역                평균 배송 지연(분)          셀 가중치
Static               Northeast                         19.687507651              1
Static               Midwest                          17.9871067398              1
Static               West                             16.8240210086              1
Dynamic              Northeast                        15.9537535365              1
Dynamic              Midwest                          14.4415169858              1
Dynamic              West                             12.8636389493              1
AIOpt                Northeast                        12.6143811724              1
AIOpt                Midwest                           10.893885771              1
AIOpt                West                               9.635351403              1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. 전체 설계에 대한 표본크기

설계를 확보한 후, GLMPOWER에게 80%와 90% 검정력에서 **총 표본크기를 계산**(`NTOTAL = .`)하도록 요청합니다. `MODEL` 문은 상호작용을 포함한 이원 배치(`routing_strategy*depot_region`)를 지정하고, `WEIGHT cell_n`은 균형 셀 가중치 프로필을 선언하며, `STDDEV = 8`은 배송 지연의 가정된 평균제곱근오차입니다. `NFRACTIONAL`은 프로시저가 반올림 전의 정확한 소수 인원수를 보고하도록 합니다.

또한 세 가지 계획된 `CONTRAST` 비교 -- AI 최적화 대 정적, 동적 대 정적, 재경로(임의) 대 정적 -- 를 사전에 등록하여, 이 시험이 검정하도록 설계된 운영상 의미 있는 가설을 문서화합니다.

In [2]:
/* ================================================================
   경로배정 전략 및 배송거점 지역 효과에 대해 80%와 90% 검정력에
   도달하는 데 필요한 총 기사 수를 계산.
   ================================================================ */
처리 glmpower 데이터=routing_trial;
   분류 routing_strategy depot_region;
   모형 mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   라벨 routing_strategy="경로배정 전략" depot_region="배송거점 지역"
         mean_delay_min="평균 배송 지연(분)";
   가중 cell_n;
   CONTRAST "AI 최적화 대 정적"     routing_strategy -1  0  1;
   CONTRAST "동적 대 정적"         routing_strategy -1  1  0;
   CONTRAST "재경로(임의) 대 정적"  routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   제목 "지연에서 경로배정 전략 효과를 검출하기 위한 표본크기";
실행;

                                                 예시 셀 평균값: 예상 배송 지연(분)                                                  

The GLMPOWER Procedure


                  Fixed Scenario Elements                  

Item                Value                                  
------------------  ---------------------------------------
Dependent Variable  평균 배송 지연(분)                            
Source              경로배정 전략                                
Source              배송거점 지역                                
Source              경로배정 전략*배송거점 지역                        

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. 변동성과 시험 규모에 대한 검정력 민감도

표본크기 답은 사전에 정확히 알기 어려운 가정된 오차 표준편차에 좌우됩니다. 여기서는 질문을 뒤집습니다: 여러 후보 총 등록 인원(`NTOTAL = 45 90 180`)을 **고정**하고, 그럴듯한 지연 표준편차(6, 8, 10분)의 격자에 대해 **달성 검정력을 계산**(`POWER = .`)합니다. 결과 표는 민감도 지도입니다 -- 실제 경로 변동성이 예상보다 높을 경우 각 등록 계획이 얼마나 견고한지를 보여줍니다.

In [3]:
/* ================================================================
   민감도 격자: 후보 시험 규모와 그럴듯한 지연 표준편차에 걸친
   달성 검정력.
   ================================================================ */
처리 glmpower 데이터=routing_trial;
   분류 routing_strategy depot_region;
   모형 mean_delay_min = routing_strategy depot_region;
   라벨 routing_strategy="경로배정 전략" depot_region="배송거점 지역"
         mean_delay_min="평균 배송 지연(분)";
   가중 cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   제목 "변동성 및 시험 규모 시나리오별 달성 검정력";
실행;

                                                 예시 셀 평균값: 예상 배송 지연(분)                                                  

The GLMPOWER Procedure


           Fixed Scenario Elements           

Item                Value                    
------------------  -------------------------
Dependent Variable  평균 배송 지연(분)              
Source              경로배정 전략                  
Source              배송거점 지역                  

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. 등록 인원 결정을 위한 검정력 곡선

마지막으로, 예상되는 8분 표준편차에서 전략+지역 모형에 대해 총 등록 인원이 30명에서 270명까지 30명 단위로 늘어남에 따른 달성 검정력, 즉 **검정력 곡선**을 추적합니다. 이 등록 인원 격자에 대해 `POWER = .`를 계산하면 `N Total` 대 `Power` 계열의 표로 곡선이 나오며, 이를 통해 관례적인 80%와 90% 목표를 각각 어느 등록 인원에서 넘어서는지 읽어낼 수 있습니다.

In [4]:
/* ================================================================
   검정력 곡선: 30명 단위로 30명에서 270명까지 늘어난 총 등록
   기사 수 대비 달성 검정력.
   ================================================================ */
처리 glmpower 데이터=routing_trial;
   분류 routing_strategy depot_region;
   모형 mean_delay_min = routing_strategy depot_region;
   라벨 routing_strategy="경로배정 전략" depot_region="배송거점 지역"
         mean_delay_min="평균 배송 지연(분)";
   가중 cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   제목 "검정력 곡선: 달성 검정력 대 총 등록 기사 수";
실행;

                                                 예시 셀 평균값: 예상 배송 지연(분)                                                  

The GLMPOWER Procedure


           Fixed Scenario Elements           

Item                Value                    
------------------  -------------------------
Dependent Variable  평균 배송 지연(분)              
Source              경로배정 전략                  
Source              배송거점 지역                  

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. 해석 및 권고

이 분석은 운영팀에 방어 가능한 등록 계획을 제공합니다:

- **기준 규모 산정 (2절).** 8분의 잔차 표준편차를 가정할 때, 상호작용을 포함한 전체 이원 모형(전략, 지역, 그 상호작용)은 **기사 63명에서 80% 검정력**, **기사 83명에서 90% 검정력**에 도달합니다. 이탈률을 감안해 반올림하면, **기사 약 90명**의 등록 목표가 90% 임계값을 여유 있게 넘습니다.

- **민감도가 중요합니다 (3절).** 검정력은 지연 변동성에 매우 민감합니다. 기사 90명일 때 달성 검정력은 표준편차 6분에서 약 99%, 8분에서 약 87%, 10분에서 약 68%로 떨어집니다. 45명 규모의 파일럿은 변동성이 낮을 때만(표준편차 6분에서 약 81% 검정력) 충분하며, 표준편차가 10분에 이르면 심각하게 검정력이 부족합니다(약 37%). 실무적 함의는: 변동성을 낮게 유지하도록 일관되고 계측이 잘 된 경로에 투자하는 것이 기사를 추가하는 것만큼 가치가 있다는 것입니다.

- **검정력 곡선 (4절).** (상호작용 항이 없는, 민감도 스윕에 사용한 렌즈인) 전략+지역 모형에 대해 추적한 결과, 달성 검정력은 기사 30명에서 0.37, 60명에서 0.69, 90명에서 0.87, 120명에서 0.95로 상승하며 180명 이상에서는 0.99를 넘어 평탄해집니다. 곡선을 목표치와 대조하면, 80% 검정력은 기사 약 77명, 90% 검정력은 약 99명에서 나타나며 -- 이는 2절의 전체 모형 규모 산정보다 다소 높습니다. 상호작용 항을 제외하면 동일한 효과가 더 적은 모형 자유도에 걸쳐 퍼지기 때문입니다.

**권고:** 약 **기사 90명**을 등록하십시오(경로배정 전략당 30명씩, 세 배송거점 지역에 균형 있게 배정). 이는 예상되는 8분 표준편차 하에서 전체 모형의 90% 검정력을 충족하고, 더 보수적인 축소 모형 곡선에서도 약 87% 검정력을 유지하며, 단일 운영 분기 내에 실행할 수 있을 만큼 시험 규모를 작게 유지합니다.

*참고:* GLMPOWER는 관측된 결과가 아니라 예상된 *설계*를 분석합니다 -- 따라서 이 수치의 신뢰도는 예상 평균값과 표준편차에 달려 있습니다. 팀은 배송 지연 변동성에 대한 실증적 추정치가 짧은 파일럿에서 나오는 대로 규모 산정을 다시 검토해야 합니다.